In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
import os
import json
import re

# 1. 룰 룩업 테이블 (is_bundle, is_confidence, is_trend_hype만 유지)
RULES = {
    "trend_hype": [
        "재입고", "품절", "주문폭주", "폭주", "입고", "hot", "히트",
        "인기", "베스트", "실시간", "트렌드", "요즘", "대세", "누적", "돌파", "1위", "1등", "유행"
        "인스타", "화제", "대란", "셀럽", "출연", "착용","랭킹"],
    "bundle": ["1+1", "증정", "묶음", "할인", "2+1", "+", "특가", "PACK", "pack","블프","블랙프라이데이","세일","SALE"],
    "confidence": [
        "MD소장", "MD", "모델소장", "전직원구매", "추천", "강추", "pick", "PICK", "픽",
        "소장", "극찬", "원픽", "사장님", "안목", "실패없는", "실물보장", "믿고사는",
        "보장", "예쁨보장", "퀄리티보장"]
}

AXES_DEFS = {
  "trend_hype": (
    "이 축은 객관적인 수치와 대중의 반응(Social Proof)으로 '유행 중임'을 증명하여 구매욕을 자극하는 표현이다. "
    "판매량, 랭킹, 재입고 횟수 등 '얼마나 많은 군중이 선택했는가'가 핵심 신호다. "
    "예: 주문폭주, 품절임박, 재입고, 마감, 1위, 베스트, 누적판매, 셀럽착용, 인스타 대세, 한정수량."
  ),
  "bundle": (
    "가격 혜택이나 구성의 풍성함을 강조하여 가성비를 자극하는 축. "
    "1+1, 세트구성, 특가, PACK, pack 등 경제적 이득을 주는 키워드가 핵심이다."
  ),
  "confidence": (
    "이 축은 판매자나 전문가의 개인적인 안목과 주관적 확신(Expert Endorsement)을 통해 신뢰를 주는 표현이다. "
    "단순히 많이 팔리는 유행(Trend)과는 다르며, '전문가가 직접 써보고 검증했다'는 보증이 핵심 신호다. "
    "예: MD소장, 모델소장, 전직원구매, 추천, pick, PICK, 픽, 사장님 추천, 실패없는 선택, 실물보장, 예쁨보장."
    "단순히 남들이 많이 입는 유행과는 구분되는 '전문가의 보증'을 의미한다."
  )
}

THRESHOLDS = {
    'trend_hype': 0.35,
    'bundle': 0.33,
    'confidence': 0.33
}

# 2. 전처리 로직
def normalize_for_split(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[!?.()\/\[\]{}<>'\"\`~@#$%^&*+=\|\\,:;.\-_]", " ", text)
    text = re.sub(r"[^a-z0-9가-힣\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def split_keywords(text: str):
    text = normalize_for_split(text)
    essential_single_char = {"울", "핏", "면", "덤", "핫", "픽"}
    tokens = text.split()
    results = []
    for t in tokens:
        if len(t) >= 2 or t in essential_single_char:
            results.append(t)
    return results

def calculate_rule_score(keywords_in_text, axis):
    combined_raw_text = "".join(keywords_in_text).lower()
    score = 0.0

    # Confidence 키워드 존재 시 trend_hype 페널티
    has_confidence = False
    if axis == "trend_hype":
        for ck in RULES["confidence"]:
            if ck.lower() in combined_raw_text:
                has_confidence = True

    score_multiplier = 0.5 if (axis == "trend_hype" and has_confidence) else 1.0

    for kw in RULES.get(axis, []):
        target = kw.lower()
        if target in combined_raw_text:
            score = 4.0
            break

    return min(1.0, (score * score_multiplier) / 3.0)

# 3. 데이터셋 생성 메인 함수
def generate_final_pseudo_dataset(titles):
    print(f"🚀 [BGE-M3 + Rule] 라벨링 시작 (trend_hype / bundle / confidence 3개 축)...")
    model = SentenceTransformer('BAAI/bge-m3')

    axes_names = list(AXES_DEFS.keys())
    axes_embeds = model.encode(list(AXES_DEFS.values()), convert_to_tensor=True)

    word_cache = {}
    results = []

    for text in titles:
        if pd.isna(text): continue
        text_str = str(text)
        keywords = split_keywords(text_str)
        if not keywords: continue

        full_emb = model.encode(text_str, convert_to_tensor=True)
        full_sims = util.cos_sim(full_emb, axes_embeds)[0]

        uncached_kws = [k for k in keywords if k not in word_cache]
        if uncached_kws:
            new_embeds = model.encode(uncached_kws, convert_to_tensor=True)
            for kw, emb in zip(uncached_kws, new_embeds):
                word_cache[kw] = emb

        current_word_embeds = torch.stack([word_cache[k] for k in keywords])
        word_sims_matrix = util.cos_sim(current_word_embeds, axes_embeds)
        max_word_sims = torch.max(word_sims_matrix, dim=0)[0]

        row_result = {'text': text_str}
        for i, axis in enumerate(axes_names):
            sentence_sim = full_sims[i].item()
            word_sim = max_word_sims[i].item()
            rule_score = calculate_rule_score(keywords, axis)

            is_rule_hit = (rule_score >= 1.0) or (rule_score == 0 and word_sim >= 0.5)
            final_score = (0.275 * sentence_sim) + (0.275 * word_sim) + (0.45 * rule_score)

            if is_rule_hit or final_score >= THRESHOLDS[axis]:
                y = 1
            else:
                y = 0

            row_result[f'y_{axis}'] = y

        results.append(row_result)

    return pd.DataFrame(results)

# --- 경로 설정 ---
DRIVE_BASE_PATH = '/content/drive/MyDrive/종합'
INPUT_FILENAME = 'combined_product_names.json'
INPUT_PATH = os.path.join(DRIVE_BASE_PATH, INPUT_FILENAME)
OUTPUT_FILENAME = "pseudo_train_3axes.csv"
CSV_OUT_PATH = os.path.join(DRIVE_BASE_PATH, OUTPUT_FILENAME)

if not os.path.exists(DRIVE_BASE_PATH):
    os.makedirs(DRIVE_BASE_PATH)

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다: {INPUT_PATH}")

with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    product_names = json.load(f)

print(f"📂 데이터를 성공적으로 불러왔습니다. (개수: {len(product_names)}개)")

df_result = generate_final_pseudo_dataset(product_names)

df_result.to_csv(CSV_OUT_PATH, index=False, encoding='utf-8-sig')

print("-" * 30)
print(f"✅ 드라이브 저장 완료!")
print(f"📍 입력 파일: {INPUT_PATH}")
print(f"📍 출력 파일: {CSV_OUT_PATH}")
print(f"📊 최종 데이터 행 개수: {len(df_result)}개")
print(f"📊 컬럼: {list(df_result.columns)}")


In [ ]:
# ============================================
# 0) 설치 및 임포트
# ============================================
#!pip install scikit-learn==1.5.2

import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

# ============================================
# 1) 경로 및 하이퍼파라미터 설정
# ============================================
DRIVE_BASE_PATH = "/content/drive/MyDrive/종합"
PSEUDO_CSV_PATH = os.path.join(DRIVE_BASE_PATH, "pseudo_train_3axes.csv")  # 3축 pseudo 데이터

OUT_DIR = os.path.join(DRIVE_BASE_PATH, "ko-roberta-fashion-model-3axes")
os.makedirs(OUT_DIR, exist_ok=True)

# 3개 축만 학습
AXES = ["bundle", "confidence", "trend_hype"]

MODEL_NAME = "klue/roberta-base"
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ============================================
# 2) 데이터 로드 및 분리
# ============================================
df = pd.read_csv(PSEUDO_CSV_PATH)
df["text"] = df["text"].astype(str)

train_df, valid_df = train_test_split(df, test_size=0.2, random_state=SEED, shuffle=True)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(f"✅ 데이터 로드 완료! Train: {len(train_df)}, Valid: {len(valid_df)}")
print(f"📊 학습 컬럼: {[f'y_{a}' for a in AXES]}")

# ============================================
# 3) Dataset 정의
# ============================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FashionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"]

        # 3개 축 라벨 추출
        labels = torch.tensor([float(row[f"y_{a}"]) for a in AXES], dtype=torch.float32)

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": labels,
        }

train_loader = DataLoader(FashionDataset(train_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(FashionDataset(valid_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

# ============================================
# 4) Ko-RoBERTa 모델 클래스
# ============================================
class MultiLabelRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] 토큰
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits

model = MultiLabelRoBERTa(MODEL_NAME, num_labels=len(AXES)).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss()

# ============================================
# 5) 학습 루프
# ============================================
best_valid_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()

    # Validation
    model.eval()
    total_valid_loss = 0
    with torch.no_grad():
        for batch in valid_loader:
            v_input_ids = batch["input_ids"].to(DEVICE)
            v_attention_mask = batch["attention_mask"].to(DEVICE)
            v_labels = batch["labels"].to(DEVICE)

            v_logits = model(v_input_ids, v_attention_mask)
            v_loss = criterion(v_logits, v_labels)
            total_valid_loss += v_loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_valid_loss = total_valid_loss / len(valid_loader)

    print(f"Epoch [{epoch}/{EPOCHS}] Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_valid_loss:.4f}")

    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss

        model.roberta.save_pretrained(OUT_DIR)
        tokenizer.save_pretrained(OUT_DIR)
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "pytorch_model.bin"))

        print(f"⭐️ Best Model Saved! (Loss: {best_valid_loss:.4f})")


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. 모델 구조
class MultiLabelRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits

# 2. 경로 설정 및 로드
MODEL_NAME = "klue/roberta-base"
SAVE_DIR = "/content/drive/MyDrive/종합/ko-roberta-fashion-model-3axes"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = MultiLabelRoBERTa(MODEL_NAME, num_labels=3).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/pytorch_model.bin", map_location=device))
model.eval()

# ⚠️ 3개 축 순서 고정 (AXES 순서와 반드시 일치)
axes = ['bundle', 'confidence', 'trend_hype']

# =========================
# 정답(GT) 세트: text -> y_true(0/1)
# =========================
GT = {
"누적만장 [MADE/나살빠졌나?] 원조/카피주의 -3kg 흉곽말라핏! 자체제작 텐셀 또또 스냅 단추 헨리넥 긴팔 롱슬리브 티셔츠(5colors)":
    {"bundle":0, "confidence":0, "trend_hype":1},

"[울35%/찐말라보임] Monica wool V-neck knit 모니카 울 브이넥 니트":
    {"bundle":0, "confidence":0, "trend_hype":0},

"[여리여리/손등덮는] 예쁨보장, 쫀득 유넥 버튼 슬리브":
    {"bundle":0, "confidence":1, "trend_hype":0},

"[딸기우유] 핀터레스트 레이어드 단가라홀터 나시 긴팔티":
    {"bundle":0, "confidence":0, "trend_hype":0},

"*섹시인생템*데일리브이넥*브이넥 골지 딥브이넥 섹시 불금룩 데이트룩 썸남저격룩 번따룩 마실룩 데일리 일상 겨울 이너":
    {"bundle":0, "confidence":0, "trend_hype":0},

"[MADE : 모델극찬, 모델소장, 실물보장 / 여리핏] 러플 버튼 스냅 옆 셔링 오프숄더 언발 소매 긴팔 후드티":
    {"bundle":0, "confidence":1, "trend_hype":0},

"[누적,7000장 돌파][자체제작/묶음할인] 촉감천재 쫀득보들 청순섹시 오피스st 포켓 굴림 카라 셔츠티":
    {"bundle":1, "confidence":0, "trend_hype":1},

"썸남꼬시는룩 군살부각X *베이비 시스루 브이넥 니트*":
    {"bundle":0, "confidence":0, "trend_hype":0},

"(나시일체형/찾던핏/몸매대박/강츄!!!/모델소장) 미우 리본 셔링 긴팔 티셔츠 발레코어":
    {"bundle":0, "confidence":1, "trend_hype":1},

"[제작초특가/ 연청추가/ 고퀄리티버전/골반뽕/숏,기본,롱/몸매보정] 자체제작 골반뽕 부츠컷 투버튼 데님 팬츠":
    {"bundle":1, "confidence":0, "trend_hype":0},

"[5000장돌파핏] 살빠진거같은데? 모달 딥유넥 스퀘어넥 허얇골넓 긴팔 잔골지 롱슬리브 티셔츠":
    {"bundle":0, "confidence":0, "trend_hype":1},

"[사장pick/흘러내림X] 러빙 앙고라 하트넥 니트":
    {"bundle":0, "confidence":1, "trend_hype":0},

"[유튜버 서연's pick!/블프특가/주문폭주] 스톤 레이어드 브이 루즈 가디건 -cd 3color":
    {"bundle":1, "confidence":1, "trend_hype":1}
}

test_texts = list(GT.keys())

THRESH = {
    'bundle': 0.35,
    'confidence': 0.35,
    'trend_hype': 0.40,
}

def predict_probs(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
    with torch.no_grad():
        logits = model(inputs["input_ids"], inputs["attention_mask"])
        probs = torch.sigmoid(logits).detach().cpu().numpy()[0]
    return probs

# =========================
# 평가 실행
# =========================
y_true = []
y_pred = []

print("========== 샘플별 결과 ==========")

for text in test_texts:
    probs = predict_probs(text)

    pred_vec = [(1 if probs[i] >= THRESH[axes[i]] else 0) for i in range(len(axes))]
    true_vec = [GT[text][a] for a in axes]

    y_true.append(true_vec)
    y_pred.append(pred_vec)

    ok = (pred_vec == true_vec)

    print("\ntext:", text)
    print("GT  :", dict(zip(axes, true_vec)))
    print("PRED:", dict(zip(axes, pred_vec)))
    print("PROB:", {axes[i]: round(float(probs[i]), 4) for i in range(len(axes))})
    print("완전일치:", ok)

y_true = np.array(y_true, dtype=int)
y_pred = np.array(y_pred, dtype=int)

print("\n========== 전체 지표 ==========")

for i, a in enumerate(axes):
    acc = accuracy_score(y_true[:, i], y_pred[:, i])
    print(f"{a} Accuracy: {acc*100:.2f}%")

micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"\nMicro-F1: {micro:.4f}")
print(f"Macro-F1: {macro:.4f}")

print("\n[클래스별 리포트(멀티라벨)]")
print(classification_report(y_true, y_pred, target_names=axes, zero_division=0))


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# 1. 모델 구조
class MultiLabelRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_output))

# 2. 로드 및 설정 (최초 1회 실행)
MODEL_NAME = "klue/roberta-base"
SAVE_DIR = "/content/drive/MyDrive/종합/ko-roberta-fashion-model-3axes"
AXES = ['bundle', 'confidence', 'trend_hype']
THRESHOLDS = [0.35, 0.35, 0.40]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = MultiLabelRoBERTa(MODEL_NAME, num_labels=len(AXES)).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/pytorch_model.bin", map_location=device))
model.eval()

# ---------------------------------------------------------
# 3. 추론 함수
# ---------------------------------------------------------
def get_classification_result(product_name):
    """
    상품명 하나를 넣으면 is_bundle / is_confidence / is_trend_hype 결과 반환
    """
    inputs = tokenizer(
        product_name,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        logits = model(inputs["input_ids"], inputs["attention_mask"])
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    detected_labels = [AXES[i] for i, prob in enumerate(probs) if prob >= THRESHOLDS[i]]
    prob_dict = {AXES[i]: round(float(probs[i]), 4) for i in range(len(AXES))}

    return detected_labels, prob_dict

# ---------------------------------------------------------
# 4. 사용 예시
# ---------------------------------------------------------
if __name__ == "__main__":
    test_items = [
        "[1+1 특가] 베이직 반팔티",
        "[모델소장/pick] 리넨 루즈 셔츠",
        "[주문폭주/재입고] 트렌디 크롭 니트",
        "기본 무지 티셔츠",
    ]

    for item in test_items:
        labels, probs = get_classification_result(item)
        print(f"\n입력: {item}")
        print(f"분류 결과: {labels}")
        print(f"확률: {probs}")
